In [ ]:
%pip install uncertainties

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

data = {
    'high_gain (mV)':[1.472, 1.221, 1.237],
    'a7 (V)':[2.173, 2.173, 2.173]
}
df = pd.DataFrame(data)
current_coefficients = np.array([[4700, 0], [0, 1], [1, 0], [1, -1]])
voltage_coefficients = np.linalg.inv(np.array([[4701, 4701], [-10001, 14702]]))
voltages = [3.281, 4.555]
currents = voltage_coefficients @ voltages
print(f'Expected I1: {currents[0]} \n Expected I2: {currents[0] - currents[1]} \n Expected I3: {currents[1]}')
expected_output = current_coefficients @ currents
print(expected_output)

Expected I1: 0.00023098668295065434 
 Expected I2: -0.00023596324333076955 
 Expected I3: 0.0004669499262814239
[ 1.08563741e+00  4.66949926e-04  2.30986683e-04 -2.35963243e-04]


In [ ]:
from uncertainties import ufloat
from uncertainties import unumpy as unp

# Define components with their uncertainties (Value, Absolute Error)
# 1% error for resistors (e.g., 1% of 4700 is 47)
R_L = ufloat(4701, 47.01)   # Left branch (4.7k + 1)
R_M = ufloat(4701, 47.01)   # Middle branch (4.7k + 1)
R_R = ufloat(10001, 100.01) # Right branch (10k + 1)

# Source voltages with the 0.004V standard deviation from your data table
V1 = ufloat(3.281, 0.004)
V2 = ufloat(4.555, 0.004)

# Rebuild your exact resistance matrix with uncertainties
R_matrix = unp.matrix([
    [R_L, R_M],
    [-R_R, R_R + R_M]
])

voltages = unp.matrix([[V1], [V2]])

# Solve for currents using the inverse matrix
currents = R_matrix.I * voltages

# Extract variables based on your previous equations
I1 = currents[0,0]
I3 = currents[1,0]
I2 = I1 - I3

# Print the results with their calculated +/- errors
print(f"Expected I1: {I1} A")
print(f"Expected I2: {I2} A")
print(f"Expected I3: {I3} A")

Expected I1: 0.0002310+/-0.0000026 A
Expected I2: -0.0002360+/-0.0000022 A
Expected I3: 0.0004669+/-0.0000031 A


In [ ]:
R_L = ufloat(4701, 47.01)
R_M = ufloat(4701, 47.01)
R_R = ufloat(10001, 100.01)

# Measured source voltages
V1 = ufloat(3.281, 0.004)
V2 = ufloat(4.555, 0.004)

# Rebuild exact resistance matrix
R_matrix = unp.matrix([
    [R_L, R_M],
    [-R_R, R_R + R_M]
])

voltages = unp.matrix([[V1], [V2]])

# Base loop currents: [i1, i3]
loop_currents = R_matrix.I * voltages


# --- 2. NEW TRANSFORMATION MATRIX FOR ALL VALUES ---

# Define the individual components for the voltage drop calculations
# (1% tolerance applied to the 4.7k and 10k resistors)
R1 = ufloat(4700, 47)
R2 = ufloat(10000, 100)
R3 = ufloat(4700, 47)

# Build the transformation matrix exactly as shown in the image
transform_matrix = unp.matrix([
    [1, 0],
    [R1, 0],
    [1, -1],
    [R2, -R2],
    [0, 1],
    [0, R3]
])

# Multiply transformation matrix by the [i1, i3] vector
# This calculates [I1, VR1, I2, VR2, I3, VR3] all at once
final_results = transform_matrix @ loop_currents
a7 = 3.3 - ufloat(4700, 47) * final_results[0,0]
# Print out the results with their propagated uncertainties
print(f"I1:  {final_results[0,0]} A")
print(f"VR1: {final_results[1,0]} V")
print(f"I2:  {final_results[2,0]} A")
print(f"VR2: {final_results[3,0]} V")
print(f"I3:  {final_results[4,0]} A")
print(f"VR3: {final_results[5,0]} V")
print(f"Expected reading at A7: {a7} V")

I1:  0.0002310+/-0.0000026 A
VR1: 1.086+/-0.016 V
I2:  -0.0002360+/-0.0000022 A
VR2: -2.360+/-0.032 V
I3:  0.0004669+/-0.0000031 A
VR3: 2.195+/-0.026 V
Expected reading at A7: 2.214+/-0.016 V


In [ ]:
VR1_obsv = ufloat(0.000233, 0.000)
VR2_obsv = ufloat(-0.000249, 0.000001)
VR3_obsv = ufloat(0.000484, 0.000)

i1_obsv = VR1_obsv / ufloat(1, 0.01)
i2_obsv = VR2_obsv / ufloat(1, 0.01)
i3_obsv = VR3_obsv / ufloat(1, 0.01)

def agrees(a, b):
  zero = a-b
  if abs(zero) <= 2 * zero.s:
    return "agree"
  return "do not agree"

print(f"i1: {i1_obsv} A, {agrees(i1_obsv, final_results[0,0])}")
print(f"i2: {i2_obsv} A, {agrees(i2_obsv, final_results[2,0])}")
print(f"i3: {i3_obsv} A, {agrees(i3_obsv, final_results[4,0])}")
print(f"a7: {ufloat(2.173, 0.004)} V, {agrees(ufloat(2.173, 0.004), ufloat(2.214, 0.016))}")

observed_currents = unp.matrix([i1_obsv, i3_obsv])
print(f"\n")
final_results_2 = transform_matrix @ observed_currents.T
print(f"VR1: {final_results_2[1,0]} V, {agrees(final_results_2[1,0], final_results[1,0])}")
print(f"VR2: {final_results_2[3,0]} V, {agrees(final_results_2[3,0], final_results[3,0])}")
print(f"VR3: {final_results_2[5,0]} V, {agrees(final_results_2[5,0], final_results[5,0])}")

i1: 0.0002330+/-0.0000023 A, agree
i2: -0.0002490+/-0.0000027 A, do not agree
i3: 0.000484+/-0.000005 A, do not agree
a7: 2.173+/-0.004 V, do not agree


VR1: 1.095+/-0.015 V, agree
VR2: -2.51+/-0.06 V, do not agree
VR3: 2.275+/-0.032 V, do not agree


/tmp/ipykernel_201/3378876408.py:11: FutureWarning: AffineScalarFunc.__abs__() is deprecated. It will be removed in a future release.
  if abs(zero) <= 2 * zero.s:
/tmp/ipykernel_201/3378876408.py:11: FutureWarning: AffineScalarFunc.__le__() is deprecated. It will be removed in a future release.
  if abs(zero) <= 2 * zero.s:
